# Notebook 05: Structural Housing Risk Modeling

**Project:** City of Boston - Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

This notebook turns the earlier exploratory analysis into a clearer modeling task:

> **Can we identify neighborhoods that are likely to experience high housing violation burden in the next complete year?**

The notebook uses two related modeling targets:

1. **Regression:** predict next-year `violations_per_1000`.
2. **Classification:** predict whether a neighborhood-year will fall into the **high-burden group** next year.

The classification task is especially useful for policy interpretation: city staff often need to know which areas should be prioritized, not whether a model can predict an exact decimal value.

This notebook also adds an external **Boston Property Assessment** dataset. It contributes structural housing-stock features such as building age, property value, living area, and housing type. This makes the project more meaningful because the model no longer relies only on past violation counts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

violations = pd.read_csv(PROCESSED_DIR / 'violations_clean.csv', low_memory=False)
violations['status_dttm'] = pd.to_datetime(violations['status_dttm'], errors='coerce')
stats = pd.read_csv(PROCESSED_DIR / 'neighborhood_stats.csv')
structural_path = EXTERNAL_DIR / 'neighborhood_structural_features.csv'
if not structural_path.exists():
    structural_path = PROCESSED_DIR / 'neighborhood_structural_features.csv'
structural = pd.read_csv(structural_path)

print(f'Violation records: {len(violations):,}')
print(f'Neighborhood stats rows: {len(stats):,}')
print(f'Structural feature neighborhoods: {structural["neighborhood"].nunique():,}')
print(f'Date range in cleaned violations: {int(violations["year"].min())}-{int(violations["year"].max())}')
structural.head()


## 1. Why Add Property Assessment Data?

The earlier model mostly used violation history. That is useful, but it mainly says: neighborhoods with high past violation burden often stay high-burden.

The property assessment data adds structural housing context:

- **Building age:** older housing stock may be more prone to code issues.
- **Property value:** lower or higher assessed values may reflect investment patterns and reporting differences.
- **Housing type:** condo-heavy, multifamily, and large-lot neighborhoods have different housing structures.
- **Density proxy:** properties per 1,000 residents helps describe housing/property concentration.

We do not use parcel-level rows directly in the model. The model unit is `neighborhood-year`, so parcel data is first aggregated to neighborhood-level structural features.


In [ ]:
structural_cols = [
    'median_building_age',
    'share_pre_1940',
    'share_pre_1900',
    'share_remodeled',
    'median_total_value',
    'median_building_value',
    'median_land_value',
    'median_living_area',
    'median_gross_area',
    'median_land_sf',
    'share_condo',
    'share_multifamily',
    'properties_per_1000_residents',
]

print('Structural features used:')
print(structural[['neighborhood'] + structural_cols].head().to_string(index=False))


## 2. Build Neighborhood-Year Panel

We use complete years only. Partial 2026 records are excluded from model training and evaluation because 2026 is incomplete and would artificially lower annual violation rates.

Each row represents one neighborhood in one feature year. The target is the next complete year's population-normalized burden.


In [ ]:
COMPLETE_YEARS = list(range(2010, 2026))
MIN_TOTAL_VIOLATIONS = 50

partial_2026_records = violations[(violations['has_neighborhood'] == True) & (violations['year'] == 2026)].copy()

eligible_neighborhoods = (
    stats.loc[stats['violation_count'] >= MIN_TOTAL_VIOLATIONS, 'neighborhood']
    .sort_values()
    .tolist()
)
population = stats[['neighborhood', 'population_2025']].dropna().copy()
population = population[population['neighborhood'].isin(eligible_neighborhoods)]

df_nbhd = violations[
    (violations['has_neighborhood'] == True)
    & (violations['year'].isin(COMPLETE_YEARS))
    & (violations['neighborhood'].isin(eligible_neighborhoods))
].copy()
df_nbhd['year'] = df_nbhd['year'].astype(int)

print(f'Modeling years: {COMPLETE_YEARS[0]}-{COMPLETE_YEARS[-1]}')
print(f'Excluded partial 2026 records from modeling: {len(partial_2026_records):,}')
print(f'Eligible neighborhoods: {len(eligible_neighborhoods)}')
print(eligible_neighborhoods)


In [ ]:
grid = pd.MultiIndex.from_product(
    [eligible_neighborhoods, COMPLETE_YEARS],
    names=['neighborhood', 'year']
).to_frame(index=False)

annual = (
    df_nbhd.groupby(['neighborhood', 'year'])
    .agg(
        violation_count=('case_no', 'size'),
        open_rate=('status', lambda x: (x == 'Open').mean()),
        pct_maintenance=('description', lambda x: x.str.contains('Maintenance', case=False, na=False).mean()),
        pct_unsafe=('description', lambda x: x.str.contains('Unsafe|Dangerous', case=False, na=False).mean()),
        unique_codes=('code', 'nunique')
    )
    .reset_index()
)

panel = (
    grid.merge(annual, on=['neighborhood', 'year'], how='left')
    .merge(population, on='neighborhood', how='left')
    .sort_values(['neighborhood', 'year'])
    .reset_index(drop=True)
)

zero_fill_cols = ['violation_count', 'open_rate', 'pct_maintenance', 'pct_unsafe', 'unique_codes']
panel[zero_fill_cols] = panel[zero_fill_cols].fillna(0)
panel['violations_per_1000'] = panel['violation_count'] / panel['population_2025'] * 1000

print(f'Neighborhood-year panel shape: {panel.shape}')
panel.head()


## 3. Lagged Features and Targets

To avoid leakage, all predictive features are available before the target year.

For example:

```text
2024 features -> 2025 target
```

The model does not use 2025 outcomes to predict 2025. It uses historical features to predict the next year.


In [ ]:
panel = panel.sort_values(['neighborhood', 'year']).copy()
grouped = panel.groupby('neighborhood', group_keys=False)

panel['next_year_violations_per_1000'] = grouped['violations_per_1000'].shift(-1)
panel['next_year_violation_count'] = grouped['violation_count'].shift(-1)

panel['lag_violations_per_1000'] = panel['violations_per_1000']
panel['lag_violation_count'] = panel['violation_count']
panel['lag_open_rate'] = panel['open_rate']
panel['lag_pct_maintenance'] = panel['pct_maintenance']
panel['lag_pct_unsafe'] = panel['pct_unsafe']
panel['lag_unique_codes'] = panel['unique_codes']
panel['lag2_violations_per_1000'] = grouped['violations_per_1000'].shift(1)
panel['rolling_3yr_rate_mean'] = grouped['violations_per_1000'].transform(
    lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
)
panel['trend_2yr_rate'] = panel['lag_violations_per_1000'] - panel['lag2_violations_per_1000']

model_df = panel.dropna(subset=[
    'next_year_violations_per_1000',
    'lag2_violations_per_1000',
    'rolling_3yr_rate_mean',
    'trend_2yr_rate'
]).copy()
model_df['target_year'] = model_df['year'] + 1

history_cols = [
    'population_2025',
    'lag_violation_count',
    'lag_violations_per_1000',
    'lag_open_rate',
    'lag_pct_maintenance',
    'lag_pct_unsafe',
    'lag_unique_codes',
    'rolling_3yr_rate_mean',
    'trend_2yr_rate',
    'year'
]

model_df = model_df.merge(structural[['neighborhood'] + structural_cols], on='neighborhood', how='left')
for col in structural_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

combined_cols = history_cols + structural_cols
structural_only_cols = structural_cols + ['year']
target_col = 'next_year_violations_per_1000'

print(f'Modeling rows: {len(model_df):,}')
print(f'Feature years: {model_df["year"].min()}-{model_df["year"].max()}')
print(f'Target years: {model_df["target_year"].min()}-{model_df["target_year"].max()}')
model_df[['neighborhood', 'year', 'target_year'] + history_cols[:4] + structural_cols[:4] + [target_col]].head()


## 4. Time-Based Split

We evaluate on future years rather than randomly mixing years.

- **Train feature years:** 2011-2021
- **Test feature years:** 2022-2024
- **Test target years:** 2023-2025

This is closer to the real use case: use historical data to prioritize future housing enforcement attention.


In [ ]:
TRAIN_END_YEAR = 2021
train_df = model_df[model_df['year'] <= TRAIN_END_YEAR].copy()
test_df = model_df[model_df['year'] > TRAIN_END_YEAR].copy()

print(f'Train rows: {len(train_df):,} ({train_df["year"].min()}-{train_df["year"].max()} feature years)')
print(f'Test rows:  {len(test_df):,} ({test_df["year"].min()}-{test_df["year"].max()} feature years)')
print(f'Test target years: {test_df["target_year"].min()}-{test_df["target_year"].max()}')


## 5. Regression: Predict Next-Year Violation Burden

First, we evaluate continuous prediction of next-year `violations_per_1000`.

We compare three feature sets:

1. **History only:** lagged violation patterns.
2. **History + property features:** lagged violation patterns plus structural housing features.
3. **Property features only:** structural housing features without violation history.

This comparison answers whether the external property data improves prediction or mainly improves interpretation.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

regression_models = {
    'Ridge Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=3,
        random_state=42
    ),
}

feature_sets = {
    'History only': history_cols,
    'History + property features': combined_cols,
    'Property features only': structural_only_cols,
}

regression_rows = []
regression_predictions = pd.DataFrame({
    'neighborhood': test_df['neighborhood'].values,
    'feature_year': test_df['year'].values,
    'target_year': test_df['target_year'].values,
    'actual_next_year_violations_per_1000': test_df[target_col].values,
    'Persistence baseline': test_df['lag_violations_per_1000'].values,
})

baseline_row = {'feature_set': 'Baseline', 'model': 'Persistence baseline'}
baseline_row.update(regression_metrics(test_df[target_col], regression_predictions['Persistence baseline']))
regression_rows.append(baseline_row)

fitted_regression_models = {}
for feature_set_name, cols in feature_sets.items():
    X_train = train_df[cols]
    y_train = train_df[target_col]
    X_test = test_df[cols]
    y_test = test_df[target_col]

    for model_name, model in regression_models.items():
        fitted = clone(model)
        fitted.fit(X_train, y_train)
        pred_col = f'{feature_set_name} - {model_name}'
        regression_predictions[pred_col] = fitted.predict(X_test)
        fitted_regression_models[pred_col] = {'model': fitted, 'features': cols}

        row = {'feature_set': feature_set_name, 'model': model_name}
        row.update(regression_metrics(y_test, regression_predictions[pred_col]))
        regression_rows.append(row)

regression_metrics_df = pd.DataFrame(regression_rows).sort_values('MAE').reset_index(drop=True)
best_regression_label = (
    regression_metrics_df[regression_metrics_df['model'] != 'Persistence baseline']
    .iloc[0]
)
best_regression_col = f"{best_regression_label['feature_set']} - {best_regression_label['model']}"

print(regression_metrics_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f'\nBest trained regression model: {best_regression_col}')
regression_metrics_df


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
plot_df = regression_metrics_df.copy()
plot_df['model_label'] = plot_df['feature_set'] + '\n' + plot_df['model']
sns.barplot(
    data=plot_df,
    x='model_label', y='MAE', hue='model_label',
    palette='Blues_r', legend=False, ax=ax
)
ax.set_title('Regression Performance: Next-Year Violation Burden')
ax.set_xlabel('')
ax.set_ylabel('MAE\n(violations per 1,000 residents)')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '13_regression_model_comparison.png', bbox_inches='tight')
plt.close()
print('Saved: 13_regression_model_comparison.png')


### 5a. Actual vs. Predicted

The scatter plot shows the best trained regression model. Downtown remains a high-burden outlier, so the lower-burden neighborhoods cluster near the bottom-left. This is a data reality, not a plotting error.


In [ ]:
best_regression_pred = regression_predictions[best_regression_col]
y_test = test_df[target_col]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, best_regression_pred, s=75, color='steelblue', alpha=0.8, zorder=5)
maxval = max(y_test.max(), best_regression_pred.max()) * 1.08
ax.plot([0, maxval], [0, maxval], color='tomato', linestyle='--', linewidth=1.5, label='Perfect prediction')

for _, row in regression_predictions.iterrows():
    label = f"{row['neighborhood']} {int(row['target_year'])}"
    ax.annotate(label,
                (row['actual_next_year_violations_per_1000'], row[best_regression_col]),
                xytext=(5, 3), textcoords='offset points', fontsize=6.5, alpha=0.7)

ax.set_xlabel('Actual next-year violations per 1,000 residents')
ax.set_ylabel('Predicted next-year violations per 1,000 residents')
ax.set_title(f'{best_regression_col}: Actual vs. Predicted Future Burden')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '14_best_regression_actual_vs_predicted.png', bbox_inches='tight')
plt.close()
print('Saved: 14_best_regression_actual_vs_predicted.png')


## 6. Classification: Identify High-Burden Neighborhood-Years

Regression is useful, but exact numeric prediction can be unstable with a small neighborhood-level dataset. A more practical question is:

> **Will this neighborhood be in the high-burden group next year?**

We define high burden using the top quartile of next-year violation rates in the training set. This threshold is learned from training data only, then applied to the held-out future years.


In [ ]:
high_burden_threshold = train_df[target_col].quantile(0.75)
train_df['high_burden_next_year'] = (train_df[target_col] >= high_burden_threshold).astype(int)
test_df['high_burden_next_year'] = (test_df[target_col] >= high_burden_threshold).astype(int)

print(f'High-burden threshold from training data: {high_burden_threshold:.3f} violations per 1,000 residents')
print(f'Train positives: {train_df["high_burden_next_year"].sum()} / {len(train_df)}')
print(f'Test positives:  {test_df["high_burden_next_year"].sum()} / {len(test_df)}')


In [ ]:
def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    }

classification_models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
    ]),
    'Random Forest Classifier': RandomForestClassifier(
        n_estimators=500,
        max_depth=4,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    ),
}

classification_rows = []
classification_predictions = pd.DataFrame({
    'neighborhood': test_df['neighborhood'].values,
    'feature_year': test_df['year'].values,
    'target_year': test_df['target_year'].values,
    'actual_high_burden': test_df['high_burden_next_year'].values,
})

persistence_class = (test_df['lag_violations_per_1000'] >= high_burden_threshold).astype(int)
classification_predictions['Persistence baseline'] = persistence_class.values
row = {'feature_set': 'Baseline', 'model': 'Persistence baseline'}
row.update(classification_metrics(test_df['high_burden_next_year'], persistence_class))
classification_rows.append(row)

fitted_classification_models = {}
for feature_set_name, cols in feature_sets.items():
    X_train = train_df[cols]
    y_train = train_df['high_burden_next_year']
    X_test = test_df[cols]
    y_test_class = test_df['high_burden_next_year']

    for model_name, model in classification_models.items():
        fitted = clone(model)
        fitted.fit(X_train, y_train)
        pred_col = f'{feature_set_name} - {model_name}'
        prob_col = f'{pred_col} probability'
        classification_predictions[pred_col] = fitted.predict(X_test)
        classification_predictions[prob_col] = fitted.predict_proba(X_test)[:, 1]
        fitted_classification_models[pred_col] = {'model': fitted, 'features': cols}

        row = {'feature_set': feature_set_name, 'model': model_name}
        row.update(classification_metrics(y_test_class, classification_predictions[pred_col]))
        classification_rows.append(row)

classification_metrics_df = pd.DataFrame(classification_rows)
classification_metrics_df = classification_metrics_df.sort_values(
    ['F1', 'Accuracy', 'Balanced Accuracy'], ascending=False
).reset_index(drop=True)
best_classifier_label = classification_metrics_df[classification_metrics_df['model'] != 'Persistence baseline'].iloc[0]
best_classifier_col = f"{best_classifier_label['feature_set']} - {best_classifier_label['model']}"

print(classification_metrics_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f'\nBest trained high-burden classifier: {best_classifier_col}')
print('\nConfusion matrix for best classifier [[TN, FP], [FN, TP]]:')
print(confusion_matrix(test_df['high_burden_next_year'], classification_predictions[best_classifier_col]))
classification_metrics_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_clf = classification_metrics_df.copy()
plot_clf['model_label'] = plot_clf['feature_set'] + '\n' + plot_clf['model']

sns.barplot(
    data=plot_clf,
    x='model_label', y='Accuracy', hue='model_label',
    palette='Greens_r', legend=False, ax=axes[0]
)
axes[0].set_title('High-Burden Classification Accuracy')
axes[0].set_xlabel('')
axes[0].set_ylabel('Accuracy')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(
    data=plot_clf,
    x='model_label', y='F1', hue='model_label',
    palette='Purples_r', legend=False, ax=axes[1]
)
axes[1].set_title('High-Burden Classification F1')
axes[1].set_xlabel('')
axes[1].set_ylabel('F1 score')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '15_high_burden_classification_performance.png', bbox_inches='tight')
plt.close()
print('Saved: 15_high_burden_classification_performance.png')


## 7. Interpreting the Model

The regression and classification results answer different questions:

- Regression estimates the approximate future burden.
- Classification identifies whether an area is likely to be high-burden.

For interpretation, we inspect the best Ridge model when available because standardized coefficients are easier to explain than tree importances.


In [ ]:
# Use the history-only Ridge model for clean coefficient interpretation if available.
interpretation_label = 'History only - Ridge Regression'
if interpretation_label in fitted_regression_models:
    interpretation_model = fitted_regression_models[interpretation_label]['model']
    interpretation_features = fitted_regression_models[interpretation_label]['features']
    coefficients = pd.Series(
        interpretation_model.named_steps['model'].coef_,
        index=interpretation_features
    ).sort_values(key=lambda s: s.abs(), ascending=False)
else:
    interpretation_label = best_regression_col
    interpretation_model = fitted_regression_models[best_regression_col]['model']
    interpretation_features = fitted_regression_models[best_regression_col]['features']
    if hasattr(interpretation_model, 'feature_importances_'):
        coefficients = pd.Series(interpretation_model.feature_importances_, index=interpretation_features).sort_values(ascending=False)
    else:
        coefficients = pd.Series(dtype=float)

fig, ax = plt.subplots(figsize=(9, 5.5))
coef_df = coefficients.head(12).reset_index()
coef_df.columns = ['feature', 'importance']
sns.barplot(data=coef_df, x='importance', y='feature', hue='feature', palette='vlag', legend=False, ax=ax)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized coefficient' if 'Ridge' in interpretation_label else 'Feature importance')
ax.set_ylabel('')
ax.set_title(f'Feature Interpretation\n({interpretation_label})')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '16_feature_interpretation.png', bbox_inches='tight')
plt.close()
print('Saved: 16_feature_interpretation.png')
print(coefficients.head(12).to_string(float_format=lambda x: f'{x:.3f}'))


## 8. 2026 Forecast and Priority View

After validating the models on complete historical years, we generate a forward-looking 2026 forecast.

Partial 2026 records are not used as ground truth. Instead:

```text
2011-2024 features -> 2012-2025 targets  (final training pairs)
2025 features      -> 2026 forecast
```

The final priority score combines:

- predicted 2026 violation burden from the best regression model
- high-burden probability from the best classifier
- 2025 unresolved/open violation rate


In [ ]:
forecast_feature_year = 2025
forecast_target_year = 2026
forecast_2026 = panel[panel['year'] == forecast_feature_year].copy()
forecast_2026 = forecast_2026.merge(structural[['neighborhood'] + structural_cols], on='neighborhood', how='left')
for col in structural_cols:
    forecast_2026[col] = forecast_2026[col].fillna(model_df[col].median())

# Refit the selected regression model on all complete historical prediction pairs.
reg_info = fitted_regression_models[best_regression_col]
final_regression_model = clone(reg_info['model'])
final_regression_model.fit(model_df[reg_info['features']], model_df[target_col])
forecast_2026['predicted_2026_violations_per_1000'] = final_regression_model.predict(forecast_2026[reg_info['features']])

# Refit the selected classifier on all complete historical prediction pairs.
clf_info = fitted_classification_models[best_classifier_col]
final_classifier_model = clone(clf_info['model'])
model_df['high_burden_next_year'] = (model_df[target_col] >= high_burden_threshold).astype(int)
final_classifier_model.fit(model_df[clf_info['features']], model_df['high_burden_next_year'])
forecast_2026['high_burden_probability_2026'] = final_classifier_model.predict_proba(forecast_2026[clf_info['features']])[:, 1]
forecast_2026['predicted_high_burden_2026'] = final_classifier_model.predict(forecast_2026[clf_info['features']])

forecast_2026['current_2025_violations_per_1000'] = forecast_2026['lag_violations_per_1000']
forecast_2026['current_2025_open_rate'] = forecast_2026['lag_open_rate']
forecast_2026['forecast_year'] = forecast_target_year

x_cut = forecast_2026['predicted_2026_violations_per_1000'].median()
y_cut = forecast_2026['current_2025_open_rate'].median()
forecast_2026['priority_group'] = np.select(
    [
        (forecast_2026['predicted_2026_violations_per_1000'] >= x_cut) & (forecast_2026['current_2025_open_rate'] >= y_cut),
        (forecast_2026['predicted_2026_violations_per_1000'] >= x_cut) & (forecast_2026['current_2025_open_rate'] < y_cut),
        (forecast_2026['predicted_2026_violations_per_1000'] < x_cut) & (forecast_2026['current_2025_open_rate'] >= y_cut),
    ],
    [
        'High forecast burden / high open rate',
        'High forecast burden / lower open rate',
        'Lower forecast burden / high open rate',
    ],
    default='Lower forecast burden / lower open rate'
)

forecast_2026[[
    'neighborhood', 'forecast_year', 'current_2025_violations_per_1000',
    'predicted_2026_violations_per_1000', 'high_burden_probability_2026',
    'predicted_high_burden_2026', 'current_2025_open_rate', 'priority_group'
]].sort_values(['predicted_high_burden_2026', 'high_burden_probability_2026'], ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=forecast_2026,
    x='predicted_2026_violations_per_1000',
    y='current_2025_open_rate',
    hue='priority_group',
    size='high_burden_probability_2026',
    sizes=(80, 220),
    ax=ax
)
ax.axvline(x_cut, color='gray', linestyle='--', linewidth=1)
ax.axhline(y_cut, color='gray', linestyle='--', linewidth=1)

for _, row in forecast_2026.iterrows():
    ax.annotate(row['neighborhood'],
                (row['predicted_2026_violations_per_1000'], row['current_2025_open_rate']),
                xytext=(5, 4), textcoords='offset points', fontsize=8, alpha=0.85)

ax.set_xlabel('Forecasted 2026 violations per 1,000 residents')
ax.set_ylabel('2025 open violation rate')
ax.set_title('2026 Neighborhood Priority Forecast\n(predicted burden, high-burden probability, and unresolved-case rate)')
ax.legend(title='', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '17_forecast_2026_priority.png', bbox_inches='tight')
plt.close()
print('Saved: 17_forecast_2026_priority.png')


## 9. Save Modeling Outputs

The saved files make the modeling results reproducible and easy to cite in the final README report.


In [ ]:
regression_metrics_df.to_csv(PROCESSED_DIR / 'modeling_regression_metrics.csv', index=False)
classification_metrics_df.to_csv(PROCESSED_DIR / 'modeling_classification_metrics.csv', index=False)
regression_predictions.to_csv(PROCESSED_DIR / 'modeling_regression_predictions.csv', index=False)
classification_predictions.to_csv(PROCESSED_DIR / 'modeling_classification_predictions.csv', index=False)
model_df.to_csv(PROCESSED_DIR / 'neighborhood_year_model_features.csv', index=False)

forecast_cols = [
    'neighborhood', 'forecast_year', 'current_2025_violations_per_1000',
    'predicted_2026_violations_per_1000', 'high_burden_probability_2026',
    'predicted_high_burden_2026', 'current_2025_open_rate', 'priority_group',
] + structural_cols
forecast_2026[forecast_cols].sort_values(
    ['predicted_high_burden_2026', 'high_burden_probability_2026'], ascending=False
).to_csv(PROCESSED_DIR / 'forecast_2026.csv', index=False)

print('Saved: data/processed/modeling_regression_metrics.csv')
print('Saved: data/processed/modeling_classification_metrics.csv')
print('Saved: data/processed/modeling_regression_predictions.csv')
print('Saved: data/processed/modeling_classification_predictions.csv')
print('Saved: data/processed/neighborhood_year_model_features.csv')
print('Saved: data/processed/forecast_2026.csv')


## Summary

This notebook reframes the project around a more meaningful housing-risk task.

Main conclusions:

- Exact regression is possible, but the dataset is small and Downtown is a strong outlier.
- The high-burden classification task is easier to explain as a policy tool: it asks which neighborhoods are likely to need attention next year.
- External property assessment data makes the project more substantively meaningful by adding structural housing features such as building age, property values, and housing type.
- Historical violation burden remains a very strong predictor. This is not a failure; it means housing violation burden is persistent across years.
- Property features may not always improve pure predictive accuracy, but they improve interpretation by connecting violation burden to the housing stock itself.
- Partial 2026 records are excluded from evaluation. The 2026 forecast uses complete 2025 features after validating models on complete historical years.

Important limitation: the model predicts observed violation burden, not true underlying housing quality. Violations are shaped by inspection practices, reporting access, tenant behavior, and city enforcement priorities.
